# page-blocks0: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('27_page-blocks0_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,85.244,80.865,90.171,73.348,99.084,86.216,90.024,98.566,83.054
smote,92.681,79.674,71.924,89.533,95.980,92.756,86.750,98.179,86.177
bsmote,94.343,77.093,65.148,94.452,94.240,94.346,83.487,98.022,85.718
ros,92.427,80.040,72.981,88.816,96.224,92.520,86.678,98.136,86.233
adasyn,94.104,75.228,62.411,94.722,93.497,94.109,83.150,97.960,84.666
casb,92.428,78.545,70.196,89.352,95.644,92.498,85.636,97.962,85.486
casb_relaxed,92.879,78.983,70.364,90.246,95.634,92.940,86.333,98.094,85.931
casb_edited,92.806,78.706,69.913,90.159,95.563,92.861,85.738,98.080,85.756
casb_ros,92.271,78.244,69.829,89.085,95.603,92.344,85.476,97.865,85.257


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,88.844,84.387,89.604,79.786,98.942,89.364,86.876,95.560,86.616
smote,92.712,82.138,76.476,88.731,96.886,92.808,83.406,95.776,87.425
bsmote,93.172,80.003,71.748,90.518,95.909,93.213,79.383,95.502,86.588
ros,92.843,81.169,74.419,89.354,96.479,92.916,82.361,95.327,87.006
adasyn,93.203,78.084,68.268,91.324,95.125,93.224,80.047,95.728,85.644
casb,92.480,81.677,76.007,88.371,96.794,92.583,83.089,95.521,87.079
casb_relaxed,93.039,81.971,75.683,89.534,96.692,93.113,83.245,95.738,87.505
casb_edited,93.781,81.165,72.985,91.505,96.122,93.814,79.775,95.847,87.473
casb_ros,92.490,81.745,76.125,88.370,96.814,92.592,83.106,95.378,87.117


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,93.140,88.809,89.986,87.752,98.870,93.311,93.223,99.086,90.975
smote,94.829,86.518,81.664,92.133,97.618,94.876,92.569,99.034,90.674
bsmote,95.004,86.352,80.967,92.577,97.506,95.042,91.708,98.989,90.678
ros,94.411,88.157,85.784,90.703,98.280,94.491,92.577,99.067,91.284
adasyn,95.199,85.536,79.067,93.293,97.160,95.227,91.011,98.985,90.367
casb,94.422,87.031,83.421,91.059,97.924,94.492,93.045,99.102,90.727
casb_relaxed,95.030,87.129,82.518,92.399,97.751,95.075,92.629,99.086,91.080
casb_edited,95.286,85.884,79.620,93.385,97.252,95.318,92.886,99.080,90.585
casb_ros,94.589,87.973,85.048,91.150,98.168,94.659,92.781,99.092,91.281


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,78.750,71.531,82.909,62.973,98.514,80.744,79.152,93.702,75.141
smote,89.175,72.256,63.529,84.341,94.362,89.351,79.801,95.491,80.716
bsmote,90.971,72.204,61.251,88.629,93.466,91.048,76.702,95.568,81.587
ros,88.786,72.625,64.674,83.271,94.708,88.990,79.838,95.583,80.705
adasyn,91.061,71.594,59.925,88.992,93.212,91.102,77.037,95.901,81.327
casb,89.809,73.173,64.258,85.412,94.514,89.963,79.159,95.497,81.491
casb_relaxed,90.186,72.545,62.875,86.661,94.025,90.343,79.320,95.559,81.365
casb_edited,91.508,73.684,62.956,89.167,93.965,91.566,79.516,95.947,82.596
casb_ros,90.525,72.715,62.563,87.287,93.965,90.626,79.165,95.544,81.620


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,86.495,81.398,88.168,75.965,98.852,87.409,87.319,96.728,83.946
smote,92.349,80.146,73.398,88.684,96.211,92.448,85.631,97.120,86.248
bsmote,93.373,78.913,69.778,91.544,95.280,93.412,82.820,97.020,86.143
ros,92.117,80.497,74.464,88.036,96.423,92.229,85.364,97.028,86.307
adasyn,93.392,77.610,67.418,92.083,94.748,93.416,82.811,97.144,85.501
casb,92.285,80.107,73.470,88.549,96.219,92.384,85.232,97.021,86.196
casb_relaxed,92.784,80.157,72.860,89.710,96.026,92.868,85.382,97.119,86.470
casb_edited,93.345,79.860,71.368,91.054,95.726,93.390,84.478,97.238,86.603
casb_ros,92.469,80.169,73.391,88.973,96.138,92.555,85.132,96.970,86.319


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,92.879,bsmote,94.343,-1.463,LOSS
1,f1,casb_relaxed,78.983,ros,80.040,-1.056,LOSS
2,precision,casb_relaxed,70.364,ros,72.981,-2.617,LOSS
3,recall,casb_relaxed,90.246,adasyn,94.722,-4.476,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,93.781,adasyn,93.203,0.578,WIN
1,f1,casb_relaxed,81.971,smote,82.138,-0.166,LOSS
2,precision,casb_ros,76.125,smote,76.476,-0.351,LOSS
3,recall,casb_edited,91.505,adasyn,91.324,0.181,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,95.286,adasyn,95.199,0.087,WIN
1,f1,casb_ros,87.973,ros,88.157,-0.184,LOSS
2,precision,casb_ros,85.048,ros,85.784,-0.735,LOSS
3,recall,casb_edited,93.385,adasyn,93.293,0.092,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,93.345,adasyn,93.392,-0.046,LOSS
1,f1,casb_ros,80.169,ros,80.497,-0.328,LOSS
2,precision,casb,73.470,ros,74.464,-0.994,LOSS
3,recall,casb_edited,91.054,adasyn,92.083,-1.029,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.861609,0.815534,0.893617,0.750000,0.989827,0.869914,0.907441,0.985431,0.838572,None
1,42,1,svm,smote,1.00,0.933271,0.771536,0.664516,0.919643,0.947101,0.933372,0.855731,0.981689,0.852403,None
2,42,1,svm,bsmote,0.50,0.943300,0.786517,0.677419,0.937500,0.949135,0.943318,0.812932,0.979427,0.864908,None
3,42,1,svm,ros,1.00,0.929228,0.769811,0.666667,0.910714,0.948118,0.929416,0.856686,0.981961,0.849520,None
4,42,1,svm,adasyn,0.75,0.951263,0.751724,0.612360,0.973214,0.929807,0.951510,0.810128,0.978501,0.851494,None
5,42,1,svm,casb,1.00,0.931264,0.760148,0.647799,0.919643,0.943032,0.931337,0.848631,0.980898,0.845706,None
6,42,1,svm,casb_relaxed,1.00,0.930761,0.757353,0.643750,0.919643,0.942014,0.930829,0.849835,0.981107,0.844057,None
7,42,1,svm,casb_edited,1.00,0.938739,0.760870,0.640244,0.937500,0.939980,0.938740,0.822652,0.979155,0.849804,None
8,42,1,svm,casb_ros,1.00,0.932268,0.765799,0.656051,0.919643,0.945066,0.932354,0.841727,0.980036,0.849034,None
9,42,1,knn,original,0.50,0.892308,0.853081,0.909091,0.803571,0.990844,0.897208,0.911685,0.963314,0.872694,None
